In [1]:
import pandas as pd
from astroquery.simbad import Simbad

mm = pd.read_csv('cf_data/pass2022_mm.csv')

# flatten into individual stars
stars = []
for _, row in mm.iterrows():
    stars.append({'name': row['comp_a'], 'mass': row['mass_a'], 'prot': row['prot_a'], 'system': f"{row['comp_a']}/{row['comp_b']}"})
    stars.append({'name': row['comp_b'], 'mass': row['mass_b'], 'prot': row['prot_b'], 'system': f"{row['comp_a']}/{row['comp_b']}"})

stars_df = pd.DataFrame(stars)
print(f'Total stars: {len(stars_df)}')
print(stars_df['name'].tolist())

Total stars: 34
['TIC 22819180', 'TIC 22819191', 'TIC 197569385', 'TIC 197570145', 'TIC 37664980', 'TIC 37664990', 'TIC 206617113', 'TIC 206617096', 'TIC 43734215', 'TIC 43789224', 'TIC 84731806', 'TIC 84731362', 'TIC 450297524', 'TIC 416857959', 'LP 68-239', '2MASS J15421300+6537051', 'LHS 1377', 'LHS 1376', '2MASS J21005492-4131438', '2MASS J21010380-4114331', 'G 115-68', 'G 115-69', 'LP 167-64', 'LP 167-63', 'LP 613-49', 'LP 613-50', 'G 116-72', 'G 116-73', 'GJ 669 A', 'GJ 669 B', 'LHS 3808', 'LHS 3809', 'GJ 49', 'GJ 51']


In [3]:
simbad = Simbad()
simbad.add_votable_fields('ids')

recovered = []
failed = []

for _, row in stars_df.iterrows():
    try:
        result = simbad.query_object(row['name'])
        if result is None:
            failed.append(row['name'])
            continue
        ids = result['ids'][0]
        dr3_id = None
        for id_str in ids.split('|'):
            if 'Gaia DR3' in id_str:
                dr3_id = int(id_str.replace('Gaia DR3', '').strip())
                break
        if dr3_id:
            recovered.append({'name': row['name'], 'mass': row['mass'], 'prot': row['prot'], 'system': row['system'], 'GaiaDR3_ID': dr3_id})
        else:
            failed.append(row['name'])
    except:
        failed.append(row['name'])

recovered_df = pd.DataFrame(recovered)
print(f'Recovered: {len(recovered_df)}')
print(f'Failed: {len(failed)}')
print(failed)

Recovered: 34
Failed: 0
[]


In [5]:
ids_str = ', '.join(str(i) for i in recovered_df['GaiaDR3_ID'].tolist())
query = f"""
SELECT source_id, bp_rp, ebpminrp_gspphot, ruwe, parallax
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""
gaia_mm = Gaia.launch_job(query).get_results().to_pandas()
gaia_mm['source_id'] = gaia_mm['source_id'].astype('Int64')
print(f'Returned: {len(gaia_mm)} rows')

Returned: 34 rows


In [6]:
mm_final = recovered_df.merge(
    gaia_mm.rename(columns={'source_id': 'GaiaDR3_ID'}),
    on='GaiaDR3_ID',
    how='left'
)

mm_final['bprp0'] = mm_final['bp_rp'] - mm_final['ebpminrp_gspphot']

print(f'Rows: {len(mm_final)}')
print(f'Missing bprp0: {mm_final["bprp0"].isna().sum()}')
print(mm_final)

Rows: 34
Missing bprp0: 15
                       name  mass    prot  \
0              TIC 22819180  0.18   0.617   
1              TIC 22819191  0.14   0.787   
2             TIC 197569385  0.17   0.443   
3             TIC 197570145  0.15   0.585   
4              TIC 37664980  0.27   1.729   
5              TIC 37664990  0.23   1.562   
6             TIC 206617113  0.22   0.490   
7             TIC 206617096  0.20   1.041   
8              TIC 43734215  0.50   1.201   
9              TIC 43789224  0.26   0.287   
10             TIC 84731806  0.31   1.993   
11             TIC 84731362  0.30   3.034   
12            TIC 450297524  0.53  22.000   
13            TIC 416857959  0.50  17.000   
14                LP 68-239  0.43   2.207   
15  2MASS J15421300+6537051  0.42   0.617   
16                 LHS 1377  0.40  11.019   
17                 LHS 1376  0.27   3.023   
18  2MASS J21005492-4131438  0.27   8.950   
19  2MASS J21010380-4114331  0.20   1.059   
20                 G 115-68 

In [8]:
mask = mm_final['bprp0'].isna() & mm_final['bp_rp'].notna()
mm_final.loc[mask, 'bprp0'] = mm_final.loc[mask, 'bp_rp']

print(f'Missing bprp0 after fix: {mm_final["bprp0"].isna().sum()}')

Missing bprp0 after fix: 0


In [ ]:
mm_final

,name,mass,prot,system,GaiaDR3_ID,bp_rp,ebpminrp_gspphot,ruwe,parallax,bprp0
0,TIC 22819180,0.18,0.617,TIC 22819180/TIC 22819191,1500827225817637504,3.193011,0.0000,1.453971,42.350580,3.193011
1,TIC 22819191,0.14,0.787,TIC 22819180/TIC 22819191,1500826882220275712,3.478862,NaN,1.443001,42.323982,3.478862
2,TIC 197569385,0.17,0.443,TIC 197569385/TIC 197570145,6586289025083317376,3.060367,0.0027,1.065400,23.756307,3.057667
3,TIC 197570145,0.15,0.585,TIC 197569385/TIC 197570145,6586283493165510272,3.194050,NaN,1.113432,23.647692,3.194050
4,TIC 37664980,0.27,1.729,TIC 37664980/TIC 37664990,2443287533259504640,2.945094,0.0005,1.154966,28.307259,2.944594
5,TIC 37664990,0.23,1.562,TIC 37664980/TIC 37664990,2443300108923745408,3.075362,NaN,1.117313,28.258650,3.075362
6,TIC 206617113,0.22,0.490,TIC 206617113/TIC 206617096,4910034311033268480,2.953927,0.0000,1.176075,28.802302,2.953927
7,TIC 206617096,0.20,1.041,TIC 206617113/TIC 206617096,4910041870175701120,3.018608,0.0006,1.307514,28.768933,3.018008
8,TIC 43734215,0.50,1.201,TIC 43734215/TIC 43789224,4500735468299667072,2.687460,0.0008,1.282061,28.582142,2.686660
9,TIC 43789224,0.26,0.287,TIC 43734215/TIC 43789224,4500722037941674496,3.177810,NaN,1.235246,28.802186,3.177810


In [11]:
mm_final['flag_high_ruwe'] = mm_final['ruwe'] > 1.4
print(mm_final[['name', 'ruwe', 'flag_high_ruwe']])
mm_final.to_csv('cf_data/pass2022_mm.csv', index=False)
print('Saved!')

                       name      ruwe  flag_high_ruwe
0              TIC 22819180  1.453971            True
1              TIC 22819191  1.443001            True
2             TIC 197569385  1.065400           False
3             TIC 197570145  1.113432           False
4              TIC 37664980  1.154966           False
5              TIC 37664990  1.117313           False
6             TIC 206617113  1.176075           False
7             TIC 206617096  1.307514           False
8              TIC 43734215  1.282061           False
9              TIC 43789224  1.235246           False
10             TIC 84731806  1.206802           False
11             TIC 84731362  1.217529           False
12            TIC 450297524  1.313132           False
13            TIC 416857959  1.359169           False
14                LP 68-239  1.477942            True
15  2MASS J15421300+6537051  1.137473           False
16                 LHS 1377  1.515242            True
17                 LHS 1376 